In [1]:
import tensorflow as tf

import tensorflow_datasets as tfds

tfds.disable_progress_bar()

In [ ]:
%pip install -q importlib-resources

datafile = "cats_vs_dogs"

split = ["train[:10%]", "train[80%:90%]", "train[90%:]"]

(train, test, valid), info = tfds.load(datafile, split = split, as_supervised = True, with_info = True)

In [3]:
labeler = info.features["label"].names

labeler = [i.capitalize() for i in labeler]

labeler

['Cat', 'Dog']

In [4]:
len(train), len(test), len(valid)

(2326, 2326, 2326)

In [5]:
input_shape = (160, 160)

inlabel = lambda i, l : (tf.image.resize(i, input_shape) / 255.0, l)

batch = 64

Resize, Shuffle, Batching

In [6]:
train = train.map(inlabel).shuffle(len(train)).batch(batch).cache()

test = test.map(inlabel).batch(batch)

valid = valid.map(inlabel).batch(batch)

In [ ]:
from tensorflow.keras.applications import MobileNet

import tensorflow.keras.layers as layers

inputs = layers.Input((160, 160, 3))

intnet = MobileNet(input_tensor = inputs, weights = 'imagenet', include_top = False)

intnet.trainable = False

In [8]:
x = layers.GlobalAveragePooling2D()(intnet.output)

output = layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs, output)

In [9]:
model.input

<KerasTensor shape=(None, 160, 160, 3), dtype=float32, sparse=False, ragged=False, name=keras_tensor>

In [10]:
model.output

<KerasTensor shape=(None, 1), dtype=float32, sparse=False, ragged=False, name=keras_tensor_87>

In [ ]:
model.compile(optimizer = "adam", loss = "binary_crossentropy", metrics = ["accuracy"])

plot = model.fit(train, validation_data = valid, epochs = 5)

In [12]:
model.evaluate(train, verbose = False)

[0.0651978999376297, 0.9772140979766846]

In [13]:
model.evaluate(valid, verbose = False)

[0.0788208618760109, 0.9729148745536804]

In [14]:
model.evaluate(valid, verbose = False)

[0.0788208618760109, 0.9729148745536804]